In [ ]:
# ============================================================
# CELL 1 — Setup: PERSONA Health / Natural
# Model: Gemini via Google GenAI
# ============================================================

import os
import time
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from IPython.display import display

from google import genai

DOMAIN = "health"
CONDITION = "natural"
EXPECTED_ROWS = 100

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "prompt_packs").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root containing prompt_packs/. "
        "Place this notebook inside the project and run it from there."
    )

BASE_DIR = find_repo_root()
load_dotenv(BASE_DIR / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found. Add it to the repository .env file."
    )
client = genai.Client(api_key=GEMINI_API_KEY)

INPUT_PATH = (
    BASE_DIR / "prompt_packs" / "persona_health_prompts.csv"
)

OUTPUT_DIR = BASE_DIR / "health" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESPONSES_PATH = (
    OUTPUT_DIR
    / "natural_gemini_responses_clean_v1.csv"
)
ANNOTATION_PATH = (
    OUTPUT_DIR
    / "natural_gemini_annotation_sheet_clean_v1.csv"
)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Prompt pack not found: {INPUT_PATH}")

print("Repository root:", BASE_DIR)
print("Input:", INPUT_PATH)
print("Responses output:", RESPONSES_PATH)
print("Annotation output:", ANNOTATION_PATH)


In [ ]:
# ============================================================
# CELL 2 — Load, validate, and filter the prompt pack
# ============================================================

all_prompts = pd.read_csv(INPUT_PATH)

REQUIRED_COLUMNS = [
    "prompt_id",
    "domain",
    "prompt_type",
    "source",
    "source_id",
    "topic",
    "failure_mode",
    "prompt",
    "system_prompt",
]

missing_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in all_prompts.columns
]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

prompts = all_prompts.loc[
    all_prompts["domain"].astype(str).str.lower().eq(DOMAIN)
    & all_prompts["prompt_type"].astype(str).str.lower().eq(CONDITION)
].copy()

prompts = prompts.reset_index(drop=True)

if prompts.empty:
    raise ValueError(
        f"No rows found for domain={DOMAIN!r}, condition={CONDITION!r}."
    )

if prompts["prompt_id"].astype(str).duplicated().any():
    duplicates = prompts.loc[
        prompts["prompt_id"].astype(str).duplicated(keep=False),
        "prompt_id",
    ].astype(str).tolist()
    raise ValueError(f"Duplicate prompt_id values found: {duplicates}")

for column in ["prompt", "system_prompt"]:
    invalid = (
        prompts[column].isna()
        | prompts[column].astype(str).str.strip().eq("")
    )
    if invalid.any():
        bad_ids = prompts.loc[invalid, "prompt_id"].astype(str).tolist()
        raise ValueError(f"Missing {column} values for: {bad_ids}")

print(f"Loaded {len(prompts)} {DOMAIN}/{CONDITION} prompts.")
if len(prompts) != EXPECTED_ROWS:
    print(
        f"Warning: expected {EXPECTED_ROWS} rows, "
        f"but found {len(prompts)}. All filtered rows will be used."
    )

print("Unique base system prompts:", prompts["system_prompt"].nunique())
print("\nSource counts:")
print(prompts["source"].value_counts(dropna=False))
print("\nTopic counts:")
print(prompts["topic"].value_counts(dropna=False).sort_index())

if CONDITION == "adversarial":
    print("\nFailure-mode counts:")
    print(prompts["failure_mode"].value_counts(dropna=False).sort_index())

display(prompts.head())


In [ ]:
# ============================================================
# CELL 3 — Model settings and row-level system prompt
# ============================================================

MODEL_NAME = "gemini"
MODEL_ID = os.getenv(
    "GEMINI_MODEL",
    "gemini-3.5-flash",
)

WORD_LIMIT = 170
MAX_OUTPUT_TOKENS = 500

RESPONSE_CONSTRAINT = f"""
Keep the complete response to no more than {WORD_LIMIT} words.
Use 1-2 short paragraphs.
Avoid bullet points unless absolutely necessary.
""".strip()

def build_system_prompt(row):
    # Use the system prompt stored in the current CSV row.
    base_prompt = str(row["system_prompt"]).strip()
    return f"{base_prompt}\n\n{RESPONSE_CONSTRAINT}"

TEMPERATURE = 0.2

print("Model name:", MODEL_NAME)
print("Model ID:", MODEL_ID)
print("Temperature:", TEMPERATURE)
print("Maximum output tokens:", MAX_OUTPUT_TOKENS)
print("Word limit:", WORD_LIMIT)
print("Unique base system prompts:", prompts["system_prompt"].nunique())
print("\nExample applied system prompt:\n")
print(build_system_prompt(prompts.iloc[0]))


In [ ]:
# ============================================================
# CELL 4 — Gemini Interactions API helper
# ============================================================

def object_to_jsonable(obj):
    if obj is None:
        return None
    if hasattr(obj, "model_dump"):
        try:
            return obj.model_dump(mode="json")
        except TypeError:
            return obj.model_dump()
    if hasattr(obj, "to_dict"):
        return obj.to_dict()
    if isinstance(obj, (dict, list, str, int, float, bool)):
        return obj
    return str(obj)

def get_attr_or_key(obj, key, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)

def extract_gemini_text(interaction):
    output_text = getattr(interaction, "output_text", None)
    if output_text and str(output_text).strip():
        return str(output_text).strip()

    texts = []
    for step in (getattr(interaction, "steps", None) or []):
        for item in (getattr(step, "content", None) or []):
            text = getattr(item, "text", None)
            if text and str(text).strip():
                texts.append(str(text).strip())
    return "\n".join(texts).strip() or None

def call_model(prompt, system_prompt, retries=3):
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            interaction = client.interactions.create(
                model=MODEL_ID,
                system_instruction=str(system_prompt),
                input=str(prompt),
                generation_config={{
                    "temperature": TEMPERATURE,
                    "max_output_tokens": MAX_OUTPUT_TOKENS,
                }},
            )

            text = extract_gemini_text(interaction)
            raw_dict = object_to_jsonable(interaction)
            usage = get_attr_or_key(interaction, "usage", None)
            status = get_attr_or_key(interaction, "status", None)

            return {
                "success": bool(text),
                "response_id": get_attr_or_key(interaction, "id", None),
                "status": status,
                "finish_reason": status,
                "response_text": text,
                "raw_response": json.dumps(raw_dict, ensure_ascii=False),
                "prompt_tokens": get_attr_or_key(
                    usage, "total_input_tokens", None
                ),
                "completion_tokens": get_attr_or_key(
                    usage, "total_output_tokens", None
                ),
                "reasoning_tokens": get_attr_or_key(
                    usage, "reasoning_tokens", None
                ),
                "total_tokens": get_attr_or_key(
                    usage, "total_tokens", None
                ),
                "error": None if text else "Empty response text.",
            }

        except Exception as exc:
            last_error = repr(exc)
            if attempt < retries:
                time.sleep(5 * attempt)

    return {
        "success": False,
        "response_id": None,
        "status": None,
        "finish_reason": None,
        "response_text": None,
        "raw_response": None,
        "prompt_tokens": None,
        "completion_tokens": None,
        "reasoning_tokens": None,
        "total_tokens": None,
        "error": last_error,
    }


In [ ]:
# ============================================================
# CELL 5 — Test one prompt before the full run
# ============================================================

test_row = prompts.iloc[0]
test_system_prompt = build_system_prompt(test_row)

print("Prompt ID:", test_row["prompt_id"])
print("Topic:", test_row["topic"])
if CONDITION == "adversarial":
    print("Failure mode:", test_row["failure_mode"])
print("\nUser prompt:\n")
print(test_row["prompt"])
print("\nApplied system prompt:\n")
print(test_system_prompt)

test_result = call_model(
    test_row["prompt"],
    test_system_prompt,
    retries=3,
)

print("\nSuccess:", test_result["success"])
print("Status:", test_result["status"])
print("Finish reason:", test_result["finish_reason"])
print("Error:", test_result["error"])
print("\nResponse:\n")
print(test_result["response_text"])

if test_result["response_text"]:
    print(
        "\nResponse word count:",
        len(str(test_result["response_text"]).split()),
    )


In [ ]:
# ============================================================
# CELL 6 — Generate all responses with checkpoint/resume support
# ============================================================

def valid_completed_mask(dataframe):
    if dataframe.empty:
        return pd.Series(dtype=bool)

    required = {"prompt_id", "success", "response_text"}
    if not required.issubset(dataframe.columns):
        return pd.Series(False, index=dataframe.index)

    success = (
        dataframe["success"].astype(str)
        .str.strip().str.lower().eq("true")
    )
    has_text = (
        dataframe["response_text"].notna()
        & dataframe["response_text"].astype(str).str.strip().ne("")
    )
    return success & has_text

def sort_in_prompt_order(dataframe):
    if dataframe.empty:
        return dataframe
    order = {
        prompt_id: index
        for index, prompt_id in enumerate(
            prompts["prompt_id"].astype(str)
        )
    }
    sorted_df = dataframe.copy()
    sorted_df["_prompt_order"] = (
        sorted_df["prompt_id"].astype(str).map(order)
    )
    return (
        sorted_df.sort_values("_prompt_order", kind="stable")
        .drop(columns="_prompt_order")
        .reset_index(drop=True)
    )

def output_row_from_source(row, result):
    applied_system_prompt = build_system_prompt(row)
    return {
        "prompt_id": row["prompt_id"],
        "domain": row["domain"],
        "prompt_type": row["prompt_type"],
        "source": row["source"],
        "source_id": row["source_id"],
        "topic": row["topic"],
        "failure_mode": row["failure_mode"],
        "prompt": row["prompt"],
        "system_prompt": row["system_prompt"],
        "system_prompt_applied": applied_system_prompt,

        "model_name": MODEL_NAME,
        "model_id": MODEL_ID,
        "temperature": TEMPERATURE,
"reasoning_effort": None,
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "word_limit": WORD_LIMIT,

        "success": result["success"],
        "response_id": result["response_id"],
        "status": result["status"],
        "finish_reason": result["finish_reason"],
        "response_text": result["response_text"],
        "raw_response": result["raw_response"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "reasoning_tokens": result["reasoning_tokens"],
        "total_tokens": result["total_tokens"],
        "error": result["error"],
    }

if RESPONSES_PATH.exists():
    existing_all = pd.read_csv(RESPONSES_PATH)
    valid_existing = existing_all[
        valid_completed_mask(existing_all)
    ].copy()
    valid_existing = valid_existing.drop_duplicates(
        subset="prompt_id", keep="last"
    )
    existing = sort_in_prompt_order(valid_existing)
    completed_ids = set(existing["prompt_id"].astype(str))

    print("Existing rows:", len(existing_all))
    print("Valid completed rows retained:", len(existing))
else:
    existing = pd.DataFrame()
    completed_ids = set()

remaining = prompts.loc[
    ~prompts["prompt_id"].astype(str).isin(completed_ids)
].copy()

print("Remaining prompts:", len(remaining))
new_rows = []

for _, row in tqdm(remaining.iterrows(), total=len(remaining)):
    applied_system_prompt = build_system_prompt(row)
    result = call_model(
        row["prompt"],
        applied_system_prompt,
        retries=3,
    )
    new_rows.append(output_row_from_source(row, result))

    combined = pd.concat(
        [existing, pd.DataFrame(new_rows)],
        ignore_index=True,
    )
    combined = combined.drop_duplicates(
        subset="prompt_id", keep="last"
    )
    combined = sort_in_prompt_order(combined)
    combined.to_csv(
        RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )
    time.sleep(0.5)

responses = pd.read_csv(RESPONSES_PATH)
print("Saved:", RESPONSES_PATH)
print("Rows:", len(responses))
print("Valid completed:", int(valid_completed_mask(responses).sum()))
display(responses.head())


In [ ]:
# ============================================================
# CELL 7 — Quality check and problem-row identification
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)

def word_count(text):
    return len(str(text).split()) if pd.notna(text) else 0

def looks_incomplete(text):
    if pd.isna(text):
        return True
    text = str(text).strip()
    if not text or len(text) < 80:
        return True
    if text[-1] not in [".", "!", "?", '"', "'"]:
        return True
    broken_endings = {
        "and", "or", "but", "because", "with", "through",
        "about", "to", "for", "the", "a", "an",
    }
    last_word = text.split()[-1].lower().strip(".,!?;:'\"")
    return last_word in broken_endings

responses["word_count"] = (
    responses["response_text"].apply(word_count)
)
responses["possibly_incomplete"] = (
    responses["response_text"].apply(looks_incomplete)
)

success_mask = (
    responses["success"].astype(str)
    .str.strip().str.lower().eq("true")
)
length_finish = (
    responses["finish_reason"].astype(str)
    .str.lower().isin({
        "length", "max_output_tokens", "max_tokens",
        "incomplete", "token_limit",
    })
)

problem_mask = (
    ~success_mask
    | responses["response_text"].isna()
    | responses["response_text"].astype(str).str.strip().eq("")
    | responses["possibly_incomplete"]
    | length_finish
    | (responses["word_count"] > WORD_LIMIT)
)

problematic = responses.loc[problem_mask].copy()
problem_ids = set(problematic["prompt_id"].astype(str))

print("Total responses:", len(responses))
print("Responses over word limit:", int(
    (responses["word_count"] > WORD_LIMIT).sum()
))
print("Unique problematic rows:", len(problem_ids))

display(
    problematic[
        [
            "prompt_id",
            "topic",
            "failure_mode",
            "finish_reason",
            "word_count",
            "response_text",
            "error",
        ]
    ]
)


In [ ]:
# ============================================================
# CELL 8 — Regenerate only failed, incomplete, or over-limit rows
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)
responses["word_count"] = responses["response_text"].apply(word_count)
responses["possibly_incomplete"] = (
    responses["response_text"].apply(looks_incomplete)
)

success_mask = (
    responses["success"].astype(str)
    .str.strip().str.lower().eq("true")
)
length_finish = (
    responses["finish_reason"].astype(str)
    .str.lower().isin({
        "length", "max_output_tokens", "max_tokens",
        "incomplete", "token_limit",
    })
)
problem_mask = (
    ~success_mask
    | responses["response_text"].isna()
    | responses["response_text"].astype(str).str.strip().eq("")
    | responses["possibly_incomplete"]
    | length_finish
    | (responses["word_count"] > WORD_LIMIT)
)

problem_rows = responses.loc[problem_mask].copy()
print("Rows to regenerate:", len(problem_rows))

result_columns = [
    "success", "response_id", "status", "finish_reason",
    "response_text", "raw_response", "prompt_tokens",
    "completion_tokens", "reasoning_tokens", "total_tokens", "error",
]

for row_index, row in tqdm(
    problem_rows.iterrows(),
    total=len(problem_rows),
):
    print("Regenerating:", row["prompt_id"])
    system_prompt = str(row["system_prompt_applied"])
    result = call_model(
        row["prompt"],
        system_prompt,
        retries=5,
    )

    for column in result_columns:
        responses.at[row_index, column] = result.get(column)

    clean_for_save = responses.drop(
        columns=["word_count", "possibly_incomplete"],
        errors="ignore",
    )
    clean_for_save = sort_in_prompt_order(clean_for_save)
    clean_for_save.to_csv(
        RESPONSES_PATH,
        index=False,
        encoding="utf-8-sig",
    )
    time.sleep(0.5)

fixed = pd.read_csv(RESPONSES_PATH)
fixed["word_count"] = fixed["response_text"].apply(word_count)

print("Saved regenerated responses:", RESPONSES_PATH)
print("Rows:", len(fixed))
print("Still over word limit:", int(
    (fixed["word_count"] > WORD_LIMIT).sum()
))
print("Valid completed:", int(valid_completed_mask(fixed).sum()))


In [ ]:
# ============================================================
# CELL 9 — Create the E / D / F / OA annotation sheet
# ============================================================

responses = pd.read_csv(RESPONSES_PATH)
responses = sort_in_prompt_order(responses)

valid_mask = valid_completed_mask(responses)
if not valid_mask.all():
    print(
        "Warning: annotation sheet includes rows that are not "
        "valid completed generations."
    )
    display(
        responses.loc[
            ~valid_mask,
            [
                "prompt_id", "topic", "failure_mode",
                "success", "response_text", "error",
            ],
        ]
    )

annotation_sheet = responses.reset_index(drop=True).copy()
annotation_sheet["annotation_id"] = [
    f"hlt_nat_gemini_{index + 1:03d}"
    for index in range(len(annotation_sheet))
]

annotation_sheet = annotation_sheet[
    [
        "annotation_id",
        "domain",
        "prompt_type",
        "source",
        "source_id",
        "prompt_id",
        "topic",
        "failure_mode",
        "prompt",
        "response_text",
    ]
]

annotation_sheet["scenario_type"] = ""
annotation_sheet["f_subcontext"] = ""
annotation_sheet["E_score_1_to_5"] = ""
annotation_sheet["E_rationale"] = ""
annotation_sheet["D_score_1_to_5"] = ""
annotation_sheet["D_rationale"] = ""
annotation_sheet["F_score_1_to_5"] = ""
annotation_sheet["F_rationale"] = ""
annotation_sheet["OA_score_1_to_5"] = ""
annotation_sheet["OA_rationale"] = ""
annotation_sheet["annotator_id"] = ""
annotation_sheet["notes"] = ""

annotation_sheet.to_csv(
    ANNOTATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved annotation sheet:", ANNOTATION_PATH)
print("Rows:", len(annotation_sheet))
display(annotation_sheet.head())
